# WP13 — Causal Safety Analysis
**Prometheus v0.97**

Answers *why* a plan is unsafe — not merely *that* it is — using do-calculus
interventions and counterfactual search:

1. **CausalSafetyGraph** — finite-difference ACE for each feature dimension
2. **Causal attribution** — which features drive unsafe outcomes and by how much
3. **CounterfactualSafetyChecker** — minimum-edit counterfactual that achieves safety
4. **WP10/WP12 integration** — causal provenance added to Debate and Hacking verdicts
5. **Full benchmark** — 4 scenarios: ACE accuracy, CF success rate, rank correlation, integration

**References**: Pearl (2009); Peters et al. (2017); Fawkes et al. (2022)


In [ ]:
import sys, os
if 'google.colab' in sys.modules:
    os.system('pip install scipy -q')
    os.system('git clone https://github.com/prometheus-ai/Prometheus_v0_PoC /content/Prometheus_v0_PoC 2>/dev/null || true')
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
else:
    sys.path.insert(0, os.path.abspath('..'))

import warnings; warnings.filterwarnings('ignore')
print('Environment ready.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import spearmanr

from prometheus.value_learning import ValueLearningAgent
from prometheus.causal_safety import (
    CausalSafetyGraph,
    CounterfactualSafetyChecker,
    CausalSafetyAnalyser,
    batch_analyse,
    causal_safety_summary_table,
)
from benchmarks.causal_safety_benchmark import (
    CausalSafetyBenchmark,
    _make_agent,
    _make_linear_reward,
    FEATURE_NAMES,
    N_FEATS,
)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

rng   = np.random.default_rng(42)
agent = _make_agent(seed=0)
print(f'ValueLearningAgent ready: {agent}')

---
## 1 — Causal Safety Graph: ACE per Feature

In [ ]:
# Controlled experiment: known linear reward
true_weights = rng.standard_normal(N_FEATS) * rng.uniform(0.5, 2.5, size=N_FEATS)
reward_fn    = _make_linear_reward(true_weights)

graph = CausalSafetyGraph(
    reward_fn     = reward_fn,
    feature_names = FEATURE_NAMES,
    delta         = 0.1,
    n_mc_samples  = 30,
)

# Average ACE over 10 input points
ace_acc = np.zeros(N_FEATS)
for trial in range(10):
    feats = rng.standard_normal(N_FEATS)
    graph.build(feats)
    ace_acc += graph.causal_influence_matrix()
mean_ace = ace_acc / 10.0

print(graph.summary())
print(f'\nSpearman ρ(|ACE|, |w|) = {spearmanr(np.abs(mean_ace), np.abs(true_weights)).statistic:.3f}')

# Bar chart: true weights vs estimated ACE
x  = np.arange(N_FEATS)
w  = 0.35
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(x - w/2, true_weights, w, label='True weight w_i',   color='#3498db', alpha=0.85)
ax.bar(x + w/2, mean_ace,     w, label='Estimated ACE_i',   color='#e74c3c', alpha=0.85)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(FEATURE_NAMES, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Value')
ax.set_title('True Weights vs Estimated Average Causal Effect (ACE)', fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

---
## 2 — Causal Attribution: Which Features Drive Unsafe Outcomes?

In [ ]:
analyser = CausalSafetyAnalyser(
    reward_fn        = agent.get_reward,
    feature_names    = FEATURE_NAMES,
    safety_threshold = 0.0,
    delta            = 0.1,
    n_mc_samples     = 20,
    cf_step_size     = 0.4,
    cf_max_steps     = 50,
)

# Find one unsafe and one safe plan
unsafe_feats = safe_feats = None
for _ in range(2000):
    f = rng.standard_normal(N_FEATS)
    r = float(agent.get_reward(f))
    if r < 0.0 and unsafe_feats is None:  unsafe_feats = f
    if r >= 0.0 and safe_feats  is None:  safe_feats   = f
    if unsafe_feats is not None and safe_feats is not None:
        break

for label, feats in [('SAFE', safe_feats), ('UNSAFE', unsafe_feats)]:
    report = analyser.analyse(feats, run_counterfactual=True)
    print(f'\n[{label}]')
    print(report.summary())

# Visualise ACE for the unsafe plan
report_unsafe = analyser.analyse(unsafe_feats, run_counterfactual=False)
ace_vals = np.array([e.weight for e in report_unsafe.causal_edges])
colors   = ['#e74c3c' if v > 0 else '#2ecc71' for v in ace_vals]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(FEATURE_NAMES, ace_vals, color=colors, edgecolor='white')
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('Causal Effect (ACE)')
ax.set_title('Causal Attribution — Unsafe Plan (▲ harmful, ▼ protective)', fontweight='bold')
harm_patch = mpatches.Patch(color='#e74c3c', label='Harmful (ACE > 0)')
prot_patch = mpatches.Patch(color='#2ecc71', label='Protective (ACE < 0)')
ax.legend(handles=[harm_patch, prot_patch])
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

---
## 3 — Counterfactual Safety: Minimum-Edit Fix

In [ ]:
# Collect unsafe plans
unsafe_plans = []
for _ in range(3000):
    f = rng.standard_normal(N_FEATS)
    if float(agent.get_reward(f)) < 0.0:
        unsafe_plans.append(f)
    if len(unsafe_plans) >= 15:
        break

print(f'Collected {len(unsafe_plans)} unsafe plans.\n')
print(f'{"Plan":>5} {"Orig reward":>12} {"CF reward":>10} {"CF safe?":>9} {"N edits":>8}')
print('-' * 50)

n_edits_list = []
success = 0
for i, f in enumerate(unsafe_plans[:10]):
    report = analyser.analyse(f, run_counterfactual=True, use_minimal_cf=True)
    cf     = report.counterfactual
    if cf:
        n_edits_list.append(cf.n_edits)
        if cf.counterfactual_safe: success += 1
        print(f'{i:>5} {cf.original_reward:>12.4f} {cf.counterfactual_reward:>10.4f} '
              f'{str(cf.counterfactual_safe):>9} {cf.n_edits:>8}')

print(f'\nSuccess rate: {success}/{min(10, len(unsafe_plans))} '
      f'({success/min(10,len(unsafe_plans))*100:.0f}%)')
if n_edits_list:
    print(f'Mean edits: {np.mean(n_edits_list):.1f}, Min: {min(n_edits_list)}')

---
## 4 — WP10/WP12 Verdict Integration

In [ ]:
# Simulate verdicts from WP10 (Debate) and WP12 (Reward Hacking)
class MockDebateVerdict:
    winner = 'opponent'
    is_safe = False
    confidence = 0.85
    def summary(self): return f'winner={self.winner}, safe={self.is_safe}, conf={self.confidence:.2f}'

class MockHackingVerdict:
    hacking_detected = True
    penalty_applied  = 0.4
    description      = 'Tampering(weight_change)'
    def summary(self): return f'hacking={self.hacking_detected}, penalty={self.penalty_applied:.2f}'

class MockFVVerdict:
    is_safe          = False
    violation_type   = 'forbidden_import'
    severity         = 9

feats  = unsafe_plans[0] if unsafe_plans else rng.standard_normal(N_FEATS)
report = analyser.analyse(
    feats,
    debate_verdict   = MockDebateVerdict(),
    hacking_verdict  = MockHackingVerdict(),
    fv_verdict       = MockFVVerdict(),
)
print(report.summary())

---
## 5 — Full Benchmark: 4 Scenarios

In [ ]:
bench   = CausalSafetyBenchmark(seed=42)
results = bench.run_all()
print(bench.summary_table(results))

In [ ]:
ace_r  = next(r for r in results if r.scenario == 'ace_accuracy')
cf_r   = next(r for r in results if r.scenario == 'counterfactual_success')
rho_r  = next(r for r in results if r.scenario == 'causal_rank_correlation')
int_r  = next(r for r in results if r.scenario == 'integration')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# (A) ACE accuracy: sign accuracy + Spearman ρ
ax = axes[0]
bars = ax.bar(['Sign accuracy', 'Spearman ρ'],
              [ace_r.metrics['sign_accuracy'], ace_r.metrics['spearman_rho']],
              color=['#3498db', '#9b59b6'], edgecolor='white')
for bar, v in zip(bars, [ace_r.metrics['sign_accuracy'], ace_r.metrics['spearman_rho']]):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.2f}',
            ha='center', fontweight='bold')
ax.axhline(0.7, color='gray', ls='--', lw=1, label='Target ≥0.7')
ax.set_ylim(0, 1.2)
ax.set_title('(A) ACE Accuracy', fontweight='bold')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# (B) Counterfactual success
ax2 = axes[1]
n_edits = cf_r.extra['n_edits_distribution']
ax2.hist(n_edits, bins=10, color='#2ecc71', edgecolor='white', alpha=0.85)
ax2.axvline(cf_r.metrics['mean_n_edits'], color='red', ls='--', lw=1.5,
            label=f"mean={cf_r.metrics['mean_n_edits']:.1f}")
ax2.set_xlabel('Edits to achieve safety')
ax2.set_ylabel('Count')
ax2.set_title(f"(B) CF Success {cf_r.metrics['success_rate']*100:.0f}%", fontweight='bold')
ax2.legend(fontsize=9)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

# (C) Causal rank correlation
ax3 = axes[2]
true_w  = np.array(rho_r.extra['true_weights'])
est_ace = np.array(rho_r.extra['mean_abs_ace'])
ax3.scatter(np.abs(true_w), est_ace, color='#e74c3c', alpha=0.8, s=80)
ax3.set_xlabel('|True weight|')
ax3.set_ylabel('|Estimated ACE|')
ax3.set_title(f"(C) Rank ρ={rho_r.metrics['spearman_rho']:.2f}", fontweight='bold')
ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)

# (D) Integration metrics
ax4 = axes[3]
metrics = ['completeness', 'cf_success\n(unsafe)', 'ms/plan\n(÷100)']
vals    = [
    int_r.metrics['completeness_rate'],
    int_r.metrics['cf_success_on_unsafe'],
    int_r.metrics['ms_per_plan'] / 100,
]
bars4 = ax4.bar(metrics, vals, color=['#3498db', '#2ecc71', '#e74c3c'], edgecolor='white')
for bar, v, raw in zip(bars4, vals, [int_r.metrics['completeness_rate'],
                                      int_r.metrics['cf_success_on_unsafe'],
                                      int_r.metrics['ms_per_plan']]):
    label = f'{raw:.0f}ms' if 'ms' in bar.get_label() else f'{raw:.2f}'
    ax4.text(bar.get_x() + bar.get_width()/2, v + 0.01,
             f"{raw:.0f}ms" if bar == bars4[-1] else f"{raw:.2f}",
             ha='center', fontweight='bold', fontsize=9)
ax4.set_title('(D) Integration Quality', fontweight='bold')
ax4.spines['top'].set_visible(False); ax4.spines['right'].set_visible(False)

fig.suptitle('Causal Safety Benchmark — Prometheus v0.97 (WP13)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

---
## Summary

| Property | Mechanism | Result |
|----------|-----------|--------|
| ACE sign accuracy | Finite-difference over 30 MC samples | **≥ 70% correct signs** |
| Rank correlation | Spearman ρ(|ACE|, |w|) | **ρ ≥ 0.60** |
| CF success rate | Greedy ACE-guided feature edits | **≥ 75% of unsafe plans fixable** |
| Report completeness | All fields populated per plan | **≥ 95%** |
| WP10/WP12/WP7 integration | Verdict summaries embedded in report | **transparent provenance** |
| Latency | Full analysis per plan | **< 1 s / plan** |

**Test coverage**: 90 tests, all passing (`pytest tests/test_causal_safety.py -v`)

**Key design**:
- `CausalSafetyGraph` uses finite-difference ACE — no LLM calls, fully deterministic
- `CounterfactualSafetyChecker` tries single-feature edits first (`minimal_counterfactual`)
  before falling back to greedy multi-feature search
- `CausalSafetyAnalyser` is a single API that composes graph + CF + verdict integration
- ACE direction: positive = reward increases with feature (harmful proxy); negative = protective

**Files**:
- `prometheus/causal_safety.py` — CausalSafetyGraph, CounterfactualSafetyChecker, CausalSafetyAnalyser
- `benchmarks/causal_safety_benchmark.py` — 4 scenarios (linear controlled + agent-based)
- `tests/test_causal_safety.py` — 90-test suite
- `notebooks/wp13_causal_safety_demo.ipynb` — this notebook
